# APL Array Notation (APLAN)

> By relieving the brain of all unnecessary work, a good notation sets it free to concentrate on more advanced problems, and, in effect, increases the mental power of the race.
\--_A. N. Whitehead, An Introduction to Mathematics_ [@whitehead1911intro]

Other resources:

- [Dyalog's docs](https://dyalog.github.io/documentation/20.0/programming-reference-guide/introduction/arrays/array-notation/)
- [APL Wiki](https://aplwiki.com/wiki/Array_notation)
- Phil Last's presentation: [Dyalog 15](https://dyalog.tv/Dyalog15/?v=9-HAvTMhYao)
- Adám Brudzewsky presentations [Dyalog 17](https://dyalog.tv/Dyalog17/?v=CRQNzL8cUQE) [Dyalog 18](https://dyalog.tv/Dyalog18/?v=GAdQuOtPcfM) [Dyalog 20](https://dyalog.tv/Dyalog20/?v=5drncJiWOM4)

Before we go too deep we should take a step back and consider how we actually construct arrays. If you come from pretty much any programming language from the Algol lineage, creating array and vector literals look remarkably similar, most likely something like Python's

```python
v = [3, 1, 4, 1]
a = [
    [3, 1, 4, 1],
    [5, 9, 2, 6]
]
```

In [24]:
⎕IO ← 0
]box on
]rows on

Was ON

Was ON

In APL, traditionally we often _reshape_ the _ravel_ to achieve the same effect -- we'll discuss both concepts later, but in essence it means that you provide the data and in a separate step mould it to the dimensions you want it to be. If we take the Python example above, in APL, you'd normally write

In [25]:
2 4⍴3 1 4 1 5 9 2 6

3 1 4 1
5 9 2 6

Two rows of four elements each from `3 1 4 1 5 9 2 6`. This is certainly compact, and when you learn APL, you soon start to "see" arrays specified like that. However, there are some obvious drawbacks, too. If the data part is long, you can't easily split it up on separate lines in your code. You can't (easily) add comments on a per-row basis in an array specification as you trivially could in Python. If you have a nested vector, you suddenly have to resort to lots of brackets:

In [26]:
(3 1 4)(1 5)(9 2 6)

┌─────┬───┬─────┐
│3 1 4│1 5│9 2 6│
└─────┴───┴─────┘

Another drawback is that if you commit a file to source control, diffs are usually on a per line basis. Change one element of the data in a long reshape expression, and the whole line gets flagged as changed, which complicates life for a reviewer.

One of the main new features of the v20 release of Dyalog was _Array Notation_ (APLAN) -- a different way to lay out array literals in code (it has other purposes, too; for later). Whilst the feature is new, it's been a long time coming. Phil Last's [presentation](https://www.dyalog.com/uploads/conference/dyalog15/presentations/U07_APL_Array_Notation.pdf) at the 2015 Dyalog User Meeting, and his subsequent [article](http://archive.vector.org.uk/art10501450) in Vector are arguably the original blueprints for Dyalog v20, even if similar ideas have been floating around the APL community since the dawn of time.

We'll be using APLAN extensively, and the purpose of this chapter is to introduce the concepts sufficiently in order for you to follow along. We'll return to APLAN in more depth in the namespaces chapter later on.

## Diamonds and newlines

APL has two syntactically equivalent forms of statement separators: newlines, and the diamond symbol, `⋄`. APLAN can use either to delineate major cells. 

The general principle is that vectors are expressed as a parenthesised APLAN expression:

In [27]:
(3 1 4 ⋄ 1 5 9 ⋄ 2 6 5 ⋄ 3 5 9) ⍝ A nested numeric vector, expressed as APLAN

┌─────┬─────┬─────┬─────┐
│3 1 4│1 5 9│2 6 5│3 5 9│
└─────┴─────┴─────┴─────┘

where each non-blank expression becomes an element in a new vector, and higher-ranked arrays expressed with square brackets, where each non-blank expression becomes a major cell (of rank≥1) in a new array. APLAN expressions can use either diamond or newline as separators -- or indeed a combination. Styles are still evolving, but in this book, we'll try to use newlines for high-rank and diamonds for vectors. 

Here's a rank-2 array:

In [28]:
[3 1 4     ⍝ A rank 2 array
 1 5 9     ⍝ with inline comments
 2 6 5     ⍝ showing π to 
 3 5 9]    ⍝ 11 decimal places

3 1 4
1 5 9
2 6 5
3 5 9

Quite nice, huh? The array is laid out visually in a way which makes its dimensions apparent. We can combine APLAN vectors and arrays freely, of course:

In [29]:
[(3 1 4 ⋄ 1 5 9)
 (2 6 5 ⋄ 3 5 9)]

┌─────┬─────┐
│3 1 4│1 5 9│
├─────┼─────┤
│2 6 5│3 5 9│
└─────┴─────┘

Using both newlines and diamonds can help visually. For the avoidance of doubt, we could instead have written the above as

In [30]:
[(3 1 4⋄1 5 9)⋄(2 6 5⋄3 5 9)]

┌─────┬─────┐
│3 1 4│1 5 9│
├─────┼─────┤
│2 6 5│3 5 9│
└─────┴─────┘

and the vector example could have been written as

In [31]:
(3 1 4
 1 5 9
 2 6 5
 3 5 9) ⍝ A nested numeric vector, expressed as APLAN

┌─────┬─────┬─────┬─────┐
│3 1 4│1 5 9│2 6 5│3 5 9│
└─────┴─────┴─────┴─────┘

but at least in this book, where convenient, we will try to stick with diamond as the separator for vectors, and newline for arrays. Saying that, APLAN also serves as a convenient way to split long strings into multiple lines in your source code, and, incidentally, an example of where you might prefer newlines instead of diamonds in a vector:

In [32]:
⍝ Baron Trump's Marvellous Underground Journey by Ingersoll Lockwood
⍝ https://www.gutenberg.org/ebooks/57426
('Bulger was not himself at all, dear friends. There was a lack-lustre look in his eyes, '
 'and his tail responded with only a half-hearted wag when I spoke to him. I say half-hearted, '
 'for I always had a notion that the other end of Bulger’s tail was fastened to his heart. '
 'His appetite, too, had gone down with his spirits; and he rarely did anything more than '
 'sniff at the dainty food which I set before him, although I tried to tempt him with fried '
 'chickens’ livers and toasted cocks’ combs—two of his favorite dishes.')

┌───────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────┐
│Bulger was not himself at all, dear friends. There was a lack-lustre look in his eyes, │and his tail responded with only a half-hearted wag when I spoke to him. I say half-hearted, │for I always had a notion that the other end of Bulger’s tail was fastened to his heart. │His appetite, too, had gone down with his spirits; and he rarely did anything more than │sniff at the dainty food which I set before him, although I tried to tempt him with fried │chickens’ livers and toasted cocks’ combs—two of his favorite dishes.│
└───────────────────────────────────────────────────────────────────────────────────────┴─────────────────────────────────────────────────────────────────────────────────────────────┴─────────────────────────────────────────────────────────────────────────────────────────┴────────────────────────────────────────────────────────────────────────────────────────┴──────────────────────────────────────────────────────────────────────────────────────────┴─────────────────────────────────────────────────────────────────────┘

At higher ranks than 2, you can still use APLAN. Here you may want to mix newlines and diamonds:

In [ ]:
[[1 1 1 ⋄ 1 1 1 ⋄ 1 1 1]
 [2 2 2 ⋄ 2 2 2 ⋄ 2 2 2]
 [3 3 3 ⋄ 3 3 3 ⋄ 3 3 3]]

1 1 1
1 1 1
1 1 1

2 2 2
2 2 2
2 2 2

3 3 3
3 3 3
3 3 3

From now on, we'll prefer APLAN when we create array literals where it makes sense -- APLAN is a syntactic change to how we write APL, mostly for the better. It has some more aces up its sleeve, which we'll encounter later on.